In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Descriptive statistics for numeric columns
print(train_data[numeric_cols].describe())

# Descriptive statistics for categorical columns
print(train_data[categorical_cols].describe())

# Check for anomalies in numeric columns
for col in numeric_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=train_data[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# Correlation matrix for numeric columns
plt.figure(figsize=(12, 10))
correlation_matrix = train_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()


     id  N_Days             Drug    Age  ... Platelets Prothrombin Stage Status
0  6703    1153          Placebo  14772  ...     236.0         9.9   3.0     CL
1  5815    1447          Placebo  14754  ...     306.0         9.5   2.0      C
2  3429    2891          Placebo  14899  ...     322.0         9.5   2.0      C
3  2405     334  D-penicillamine  22369  ...     156.0        11.0   2.0      C
4  1410    3820          Placebo  20597  ...     119.0        11.7   4.0      D

[5 rows x 20 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6324 entries, 0 to 6323
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             6324 non-null   int64  
 1   N_Days         6324 non-null   int64  
 2   Drug           6324 non-null   object 
 3   Age            6324 non-null   int64  
 4   Sex            6324 non-null   object 
 5   Ascites        6324 non-null   object 
 6   Hepatomegaly   6324 non-null   ob

Numeric Columns: Index(['id', 'N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper',
       'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin',
       'Stage'],
      dtype='object')
Categorical Columns: Index(['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Status'], dtype='object')
                id       N_Days  ...  Prothrombin        Stage
count  6324.000000  6324.000000  ...  6324.000000  6324.000000
mean   3973.850886  2023.513599  ...    10.627593     3.034946
std    2273.112160  1088.218024  ...     0.785929     0.866119
min       1.000000    41.000000  ...     9.000000     1.000000
25%    2013.750000  1230.000000  ...    10.000000     2.000000
50%    3988.500000  1831.000000  ...    10.600000     3.000000
75%    5954.250000  2689.000000  ...    11.000000     4.000000
max    7902.000000  4795.000000  ...    18.000000     4.000000

[8 rows x 13 columns]
           Drug   Sex Ascites Hepatomegaly Spiders Edema Status
count      6324  6324 

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:03:53.493 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Status'], 'Numeric': ['id', 'N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/test.csv')

# Handle missing values
fill_missing = FillMissingValue(features=numeric_cols, strategy='mean')
train_data = fill_missing.fit_transform(train_data.copy())
test_data = fill_missing.transform(test_data.copy())

# Encode categorical variables
label_encode = LabelEncode(features=categorical_cols)
train_data = label_encode.fit_transform(train_data.copy())
test_data = label_encode.transform(test_data.copy())

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
train_data = standard_scale.fit_transform(train_data.copy())
test_data = standard_scale.transform(test_data.copy())

# Display the first few rows of the processed train and test data
print(train_data.head())
print(test_data.head())


         id    N_Days  Drug       Age  ...  Platelets  Prothrombin     Stage  Status
0  1.200717 -0.800007     1 -0.969151  ...  -0.335471    -0.925848 -0.040351       1
1  0.810032 -0.529820     1 -0.974053  ...   0.465361    -1.434841 -1.195018       0
2 -0.239713  0.797225     1 -0.934562  ...   0.648408    -1.434841 -1.195018       0
3 -0.690232 -1.552673     0  1.099898  ...  -1.250708     0.473880 -1.195018       0
4 -1.127993  1.650982     1  0.617292  ...  -1.674005     1.364617  1.114316       2

[5 rows x 20 columns]
         id    N_Days  Drug       Age  ...  Platelets  Prothrombin     Stage  Status
0 -0.222994 -0.967266     0  0.605853  ...  -0.003698    -1.053097 -0.040351       0
1 -1.543755 -0.196220     0  0.605853  ...   0.865777    -0.035112  1.114316       0
2 -1.549035  0.140136     1 -0.290725  ...  -0.484197    -0.925848 -0.040351       0
3  0.413188 -0.516034     1  1.361899  ...   0.213671    -0.162360 -0.040351       2
4  0.850509 -0.742110     1  0.453065  ...

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'N_Days', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage', 'Status'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

# Assuming train_data and test_data are already preprocessed from previous steps
X = train_data.drop(columns=['id', 'Status'])
y = train_data['Status']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost classifier
model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    learning_rate=0.1,
    max_depth=6,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Predict probabilities on the validation set
y_val_pred_proba = model.predict_proba(X_val)

# Calculate log loss
logloss = log_loss(y_val, y_val_pred_proba)
print(f'Validation Log Loss: {logloss}')


Validation Log Loss: 0.4668835605318982


In [6]:
# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/test.csv')

# Preprocess the test data using the same transformations as the training data
test_data = fill_missing.transform(test_data.copy())
test_data = label_encode.transform(test_data.copy())
test_data = standard_scale.transform(test_data.copy())

# Prepare the test features
X_test = test_data.drop(columns=['id', 'Status'])

# Predict probabilities on the test data
y_test_pred_proba = model.predict_proba(X_test)

# Assuming the test data has a 'Status' column for evaluation
y_test = test_data['Status']

# Calculate the log loss on the test data
test_logloss = log_loss(y_test, y_test_pred_proba)
print(f'Test Log Loss: {test_logloss}')


Test Log Loss: 0.4464427712970508
